# Error Analysis — Diffusion-LM vs Baselines

Loads `results/generations/*.jsonl` and inspects:
1. **Per-example slot recall** — which MR slots failed to surface in the output.
2. **Length distribution** comparison.
3. **Lexical diversity vs fluency tradeoff** (distinct-2 vs BLEU).
4. **Qualitative worst cases** for each model.

Run after `scripts/run_eval.sh` has produced JSONL files.

In [ ]:
import json
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

GEN_DIR = Path('../results/generations')
models = sorted(p.stem for p in GEN_DIR.glob('*.jsonl'))
print('models:', models)

In [ ]:
def load(model):
    return [json.loads(l) for l in open(GEN_DIR / f'{model}.jsonl')]

data = {m: load(m) for m in models}
{m: len(rs) for m, rs in data.items()}

## 1. Per-slot failure rates

In [ ]:
def slot_failures(records):
    miss = Counter(); total = Counter()
    for r in records:
        pred = r['prediction'].lower()
        for slot, val in r.get('slots', {}).items():
            total[slot] += 1
            if val.lower() not in pred:
                miss[slot] += 1
    return pd.DataFrame({s: [miss[s], total[s], miss[s] / max(total[s], 1)] for s in total},
                        index=['missed', 'total', 'miss_rate']).T.sort_values('miss_rate', ascending=False)

for m in models:
    print(f'\n=== {m} ===')
    display(slot_failures(data[m]))

## 2. Output length distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for m in models:
    lens = [len(r['prediction'].split()) for r in data[m]]
    ax.hist(lens, bins=30, alpha=0.5, label=m)
ax.set_xlabel('output length (words)'); ax.set_ylabel('count'); ax.legend()
plt.show()

## 3. Diversity vs Fluency tradeoff

In [ ]:
import json
summary = {}
for m in models:
    metric_path = Path('../results/metrics') / f'{m}.json'
    if metric_path.exists():
        summary[m] = json.load(open(metric_path))
df = pd.DataFrame(summary).T
df

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for m in df.index:
    ax.scatter(df.loc[m, 'bleu'], df.loc[m, 'distinct_2'], s=80)
    ax.annotate(m, (df.loc[m, 'bleu'], df.loc[m, 'distinct_2']))
ax.set_xlabel('BLEU (fluency / fidelity)'); ax.set_ylabel('distinct-2 (lexical diversity)')
ax.set_title('Diversity-Fluency tradeoff')
plt.show()

## 4. Qualitative worst cases (lowest slot recall)

In [ ]:
def per_example_recall(r):
    slots = r.get('slots', {})
    if not slots: return 1.0
    pred = r['prediction'].lower()
    return sum(v.lower() in pred for v in slots.values()) / len(slots)

for m in models:
    print(f'\n=== {m}: 5 worst examples ===')
    ranked = sorted(data[m], key=per_example_recall)[:5]
    for r in ranked:
        print(f'MR: {r["mr_text"]}')
        print(f'REF: {r["reference"]}')
        print(f'GEN: {r["prediction"]}')
        print('---')